## Data Loading and Preprocessing

In [1]:
# load data and inspect shape
import pandas as pd

df = pd.read_parquet("data/merged/final_merged_panel.parquet")
df.shape

(12715, 33)

#### Train-Test Split

In [3]:
# split train, validation, and test sets
from sklearn.model_selection import train_test_split

# 70/15/15 split for train/validation/test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.25, random_state=42)

In [14]:
# check distribution of target variable
train_df["recession"].value_counts()

recession
0    7186
1     443
Name: count, dtype: int64

In [4]:
# separate features and target variable
X_train = train_df.drop("recession", axis=1)
y_train = train_df["recession"]

X_val = val_df.drop("recession", axis=1)
y_val = val_df["recession"]

X_test = test_df.drop("recession", axis=1)
y_test = test_df["recession"]

#### Standarization and OHE

In [18]:
# check data types of columns
num_cols = X_train.select_dtypes(include=["number"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns

print("Numerical:", list(num_cols))
print("Categorical:", list(cat_cols))

Numerical: ['year', 'wb_current_account_pct_gdp', 'wb_exports_pct_gdp', 'wb_gdp_growth', 'wb_gov_consumption_pct_gdp', 'wb_imports_pct_gdp', 'wb_inflation_cpi', 'imf_inflation_imf', 'non_null_feature_count', 'feature_coverage_pct', 'wb_current_account_pct_gdp_lag1', 'wb_current_account_pct_gdp_lag2', 'wb_exports_pct_gdp_lag1', 'wb_exports_pct_gdp_lag2', 'wb_gdp_growth_lag1', 'wb_gdp_growth_lag2', 'wb_gov_consumption_pct_gdp_lag1', 'wb_gov_consumption_pct_gdp_lag2', 'wb_imports_pct_gdp_lag1', 'wb_imports_pct_gdp_lag2', 'wb_inflation_cpi_lag1', 'wb_inflation_cpi_lag2', 'imf_inflation_imf_lag1', 'imf_inflation_imf_lag2', 'wb_current_account_pct_gdp_chg1', 'wb_exports_pct_gdp_chg1', 'wb_gdp_growth_chg1', 'wb_gov_consumption_pct_gdp_chg1', 'wb_imports_pct_gdp_chg1', 'wb_inflation_cpi_chg1', 'imf_inflation_imf_chg1']
Categorical: ['country']


In [20]:
# one-hot encode categorical variable "country"
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat = encoder.fit_transform(X_train[cat_cols])
X_val_cat = encoder.transform(X_val[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])

In [19]:
# standardize numerical features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train[num_cols])
X_val_scaled = scaler.transform(X_val[num_cols])
X_test_scaled = scaler.transform(X_test[num_cols])

#### Feature Engineering